# 01 — Análise Exploratória (EDA)

Exploração dos cinco corpora de sentimento em PT-BR: volume, distribuição de
polaridade, notas e comprimento dos textos por domínio.

> Lógica de produção vive em `src/`; este notebook é só exploração.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from src.config.paths import get_paths
from src.constants.datasets import DATASET_NAMES
from src.data.loader import prepare_domain
from src.visualization.theme import apply_theme

RANDOM_SEED = 42
apply_theme()
paths = get_paths()

## Carga e preparação por domínio

In [ ]:
frames = []
for domain in DATASET_NAMES:
    df = prepare_domain(domain, paths=paths)
    frames.append(df)
corpus = pl.concat(frames)
corpus.head()

## Volume e balanceamento de polaridade por domínio

In [ ]:
resumo = (
    corpus.group_by("dataset")
    .agg(
        pl.len().alias("n"),
        pl.col("polarity").mean().alias("taxa_positivos"),
    )
    .sort("dataset")
)
resumo

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=resumo.to_pandas(), x="dataset", y="taxa_positivos", ax=ax)
ax.set_title("Taxa de reviews positivas por domínio")
ax.set_xlabel("Domínio")
ax.set_ylabel("Proporção de positivos")
ax.axhline(0.5, ls="--", c="gray")
plt.tight_layout()

## Distribuição do comprimento dos textos

In [ ]:
com_tamanho = corpus.with_columns(pl.col("text_clean").str.len_chars().alias("n_chars"))
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=com_tamanho.filter(pl.col("n_chars") < 1000).to_pandas(),
    x="dataset",
    y="n_chars",
    ax=ax,
)
ax.set_title("Comprimento do texto limpo por domínio")
ax.set_xlabel("Domínio")
ax.set_ylabel("Nº de caracteres")
plt.tight_layout()